In [2]:
%load_ext manim

The manim module is not an IPython extension.


In [5]:
%%manim -pqm SinusUtrullning
from manim import *
import numpy as np

# Konfigurera MiKTeX-mall för säkra matematiksymboler
min_mall = TexTemplate()
min_mall.add_to_preamble(r"\usepackage{amsmath}\usepackage{amssymb}")

class SinusUtrullning(Scene):
    def construct(self):
        # 1. Huvudrubrik (Stor och tydlig överst)
        rubrik = Text("Från enhetscirkeln till sinusfunktionen", font_size=32).to_edge(UP)
        self.add(rubrik)

        # 2. Skapa enhetscirkelns miljö (till vänster)
        cirkel_center = LEFT * 3.5 + DOWN * 0.5
        
        cirkel_axlar = Axes(
            x_range=[-1.5, 1.5, 1],
            y_range=[-1.5, 1.5, 1],
            x_length=3.8,
            y_length=3.8,  # Samma längd på x och y garanterar en perfekt rund cirkel
            axis_config={"include_tip": True, "stroke_width": 2}
        ).move_to(cirkel_center)
        
        cirkel_etiketter = {
            1: MathTex("1", font_size=16),
            -1: MathTex("-1", font_size=16)
        }
        cirkel_axlar.get_x_axis().add_labels(cirkel_etiketter)
        cirkel_axlar.get_y_axis().add_labels(cirkel_etiketter)
        
        # Generera enhetscirkeln DIREKT baserat på axelns eget koordinatsystem (matematisk radie = 1)
        cirkel = cirkel_axlar.plot_parametric_curve(
            lambda t: np.array([np.cos(t), np.sin(t), 0]),
            t_range=[0, 2 * PI],
            color=GRAY,
            stroke_width=2
        )
        
        self.play(DrawBorderThenFill(cirkel_axlar), Create(cirkel), run_time=1.0)

        # 3. Skapa sinusgrafens miljö (till höger)
        # Vi sätter samma y_length=3.8 och y_range=[-1.5, 1.5] så att de fysiska höjderna matchar exakt på skärmen!
        graf_axlar = Axes(
            x_range=[0, 2 * PI + 0.3, PI / 4],
            y_range=[-1.5, 1.5, 1],
            x_length=6.0,
            y_length=3.8,
            axis_config={"include_tip": True, "stroke_width": 2}
        ).shift(RIGHT * 3.2 + DOWN * 0.5)

        x_etiketter = {
            PI / 4: MathTex(r"\frac{\pi}{4}", font_size=18, tex_template=min_mall),
            PI / 2: MathTex(r"\frac{\pi}{2}", font_size=18, tex_template=min_mall),
            3 * PI / 4: MathTex(r"\frac{3\pi}{4}", font_size=18, tex_template=min_mall),
            PI: MathTex(r"\pi", font_size=18, tex_template=min_mall),
            5 * PI / 4: MathTex(r"\frac{5\pi}{4}", font_size=18, tex_template=min_mall),
            3 * PI / 2: MathTex(r"\frac{3\pi}{2}", font_size=18, tex_template=min_mall),
            7 * PI / 4: MathTex(r"\frac{7\pi}{4}", font_size=18, tex_template=min_mall),
            2 * PI: MathTex(r"2\pi", font_size=18, tex_template=min_mall)
        }
        graf_axlar.get_x_axis().add_labels(x_etiketter)
        
        graf_y_etiketter = {
            1: MathTex("1", font_size=16),
            -1: MathTex("-1", font_size=16)
        }
        graf_axlar.get_y_axis().add_labels(graf_y_etiketter)
        
        x_lbl = MathTex("x", font_size=20).next_to(graf_axlar.get_x_axis(), RIGHT, buff=0.1)
        y_lbl = MathTex(r"y = \sin(x)", font_size=22, color=BLUE, tex_template=min_mall).next_to(graf_axlar.get_y_axis(), UP, buff=0.1)
        
        self.play(DrawBorderThenFill(graf_axlar), Write(x_lbl), Write(y_lbl), run_time=1.0)

        # 4. Skapa de statiska grund-objekten för startläget (0 radianer)
        origo = cirkel_axlar.c2p(0, 0)
        start_slut = cirkel_axlar.c2p(1, 0)
        
        visare = Line(start=origo, end=start_slut, color=WHITE, stroke_width=4)
        sin_linje_cirkel = Line(start=start_slut, end=start_slut, color=BLUE, stroke_width=5)

        info_låda = MathTex(
            r"\text{Vinkel } x = 0^\circ, \quad \sin(x) = 0.00",
            font_size=26, color=YELLOW, tex_template=min_mall
        ).to_edge(DOWN).shift(UP * 0.2)

        start_punkt = Dot(point=graf_axlar.c2p(0, 0), color=BLUE, radius=0.08)

        self.add(visare, sin_linje_cirkel, start_punkt, info_låda)
        self.wait(1.0)

        # 5. Lista över vinklarna som visaren ska "hoppa" mellan
        stopp_vinklar = [
            PI / 4, PI / 2, 3 * PI / 4, PI, 5 * PI / 4, 3 * PI / 2, 7 * PI / 4, 2 * PI
        ]

        # En temporär hjälplinje som vi flyttar runt
        aktiv_hjalp_linje = VectorizedPoint(location=ORIGIN)

        # 6. Stegvis loop runt varvet
        for vinkel in stopp_vinklar:
            # Räkna ut exakta matematiska koordinater inuti cirkelns axelsystem
            cos_v = np.cos(vinkel)
            sin_v = np.sin(vinkel)
            
            cirkel_punkt_pos = cirkel_axlar.c2p(cos_v, sin_v)
            cirkel_sin_bas = cirkel_axlar.c2p(cos_v, 0)
            
            ny_visare = Line(start=origo, end=cirkel_punkt_pos, color=WHITE, stroke_width=4)
            ny_sin_linje = Line(start=cirkel_sin_bas, end=cirkel_punkt_pos, color=BLUE, stroke_width=5)

            # Vinkelbåge genererad exakt i cirkelaxlarna
            vinkel_bage = cirkel_axlar.plot_parametric_curve(
                lambda t: np.array([0.3 * np.cos(t), 0.3 * np.sin(t), 0]),
                t_range=[0, vinkel],
                color=YELLOW,
                stroke_width=3
            )

            # 1. Animera visaren till nästa hopp
            self.play(
                Transform(visare, ny_visare),
                Transform(sin_linje_cirkel, ny_sin_linje),
                run_time=0.8
            )
            self.add(vinkel_bage)

            # Beräkna punktens exakta koordinat i grafens koordinatsystem
            graf_punkt_pos = graf_axlar.c2p(vinkel, sin_v)
            
            # 2. Rita hjälplinjen direkt mellan cirkelns punkt och grafens punkt
            # Eftersom båda punkterna använder sina respektive axlars c2p med samma sin_v, blir linjen 100% horisontell!
            ny_hjalp = DashedLine(start=cirkel_punkt_pos, end=graf_punkt_pos, color=BLUE_B, stroke_width=1.5)
            self.play(Transform(aktiv_hjalp_linje, ny_hjalp), run_time=0.5)

            # 3. Stämpla ut en punkt på grafen
            ny_punkt = Dot(point=graf_punkt_pos, color=BLUE, radius=0.08)
            self.add(ny_punkt)

            # 4. Uppdatera den matematiska texten
            ny_info = MathTex(
                rf"\text{{Vinkel }} x = {int(np.degrees(vinkel))}^\circ, \quad \sin(x) = {sin_v:.2f}",
                font_size=26, color=YELLOW, tex_template=min_mall
            ).to_edge(DOWN).shift(UP * 0.2)
            
            self.play(Transform(info_låda, ny_info), run_time=0.3)
            self.wait(1.0)
            self.remove(vinkel_bage)

        # 7. Finalen: Ta bort sista hjälplinjen och rita hela sinusgrafen i ett svep!
        self.play(FadeOut(aktiv_hjalp_linje), run_time=0.4)
        
        full_sin_graf = graf_axlar.plot(
            lambda x: np.sin(x),
            x_range=[0, 2 * PI],
            color=BLUE, stroke_width=4
        )
        
        self.play(Create(full_sin_graf), run_time=2.5)
        self.wait(2.0)

        # Tona ut allt
        self.play(*[FadeOut(mob) for mob in self.mobjects])


Manim Community v0.22.1

[11/04/26 15:59:50] INFO     Animation 0 : Partial movie file written in                   ]8;id=15997333;file://c:\Users\sanpar\AppData\Local\Programs\Python\Python312\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=15997334;file://c:\Users\sanpar\AppData\Local\Programs\Python\Python312\Lib\site-packages\manim\scene\scene_file_writer.py#173\173]8;;\
                             'c:\Users\sanpar\OneDrive - Jonkoping                                                 
                             University\JTH\Programmering\Python\enhetscirkeln till                                
                             sincosfunk\media\videos\enhetscirkeln till                                            
                             sincosfunk\720p30\partial_movie_files\SinusUtrullning\1fe0d46                         
                             c43fd855faf24cf60a705a3873c2534841026740c5ea548b6e106f25a.mp4                         
                             '                                                                                     

[11/04/26 15:59:52] INFO     Animation 1 : Partial movie file written in                   ]8;id=15997339;file://c:\Users\sanpar\AppData\Local\Programs\Python\Python312\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=15997340;file://c:\Users\sanpar\AppData\Local\Programs\Python\Python312\Lib\site-packages\manim\scene\scene_file_writer.py#173\173]8;;\
                             'c:\Users\sanpar\OneDrive - Jonkoping                                                 
                             University\JTH\Programmering\Python\enhetscirkeln till                                
                             sincosfunk\media\videos\enhetscirkeln till                                            
                             sincosfunk\720p30\partial_movie_files\SinusUtrullning\c204888                         
                             1b753dd8d2f39435e05ed17c2e29f00f230593f207ff644ec8921ebc6.mp4                         
                             '                                                                                     

[11/04/26 15:59:53] INFO     Animation 2 : Partial movie file written in                   ]8;id=15997345;file://c:\Users\sanpar\AppData\Local\Programs\Python\Python312\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=15997346;file://c:\Users\sanpar\AppData\Local\Programs\Python\Python312\Lib\site-packages\manim\scene\scene_file_writer.py#173\173]8;;\
                             'c:\Users\sanpar\OneDrive - Jonkoping                                                 
                             University\JTH\Programmering\Python\enhetscirkeln till                                
                             sincosfunk\media\videos\enhetscirkeln till                                            
                             sincosfunk\720p30\partial_movie_files\SinusUtrullning\56816c8                         
                             f5ddd93a6099bcbd0581e6208a1899679bc23a7f995a801ff07f7c464.mp4                         
                             '                                                                                     

[11/04/26 15:59:54] INFO     Animation 3 : Partial movie file written in                   ]8;id=15997351;file://c:\Users\sanpar\AppData\Local\Programs\Python\Python312\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=15997352;file://c:\Users\sanpar\AppData\Local\Programs\Python\Python312\Lib\site-packages\manim\scene\scene_file_writer.py#173\173]8;;\
                             'c:\Users\sanpar\OneDrive - Jonkoping                                                 
                             University\JTH\Programmering\Python\enhetscirkeln till                                
                             sincosfunk\media\videos\enhetscirkeln till                                            
                             sincosfunk\720p30\partial_movie_files\SinusUtrullning\623c52b                         
                             f48614e69825c5a759936b8d1f1b7d60dda5d304c46b123738dbc1bbe.mp4                         
                             '                                                                                     

[11/04/26 15:59:55] INFO     Animation 4 : Partial movie file written in                   ]8;id=15997357;file://c:\Users\sanpar\AppData\Local\Programs\Python\Python312\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=15997358;file://c:\Users\sanpar\AppData\Local\Programs\Python\Python312\Lib\site-packages\manim\scene\scene_file_writer.py#173\173]8;;\
                             'c:\Users\sanpar\OneDrive - Jonkoping                                                 
                             University\JTH\Programmering\Python\enhetscirkeln till                                
                             sincosfunk\media\videos\enhetscirkeln till                                            
                             sincosfunk\720p30\partial_movie_files\SinusUtrullning\6fbf90e                         
                             f847d94963046e46bcd76bd75803cd956c4b0f9ea2b2743cfe770cfd8.mp4                         
                             '                                                                                     

... (35 outputs left out)

                    INFO                                                                   ]8;id=15997573;file://c:\Users\sanpar\AppData\Local\Programs\Python\Python312\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=15997574;file://c:\Users\sanpar\AppData\Local\Programs\Python\Python312\Lib\site-packages\manim\scene\scene_file_writer.py#999\999]8;;\
                             File ready at 'c:\Users\sanpar\OneDrive - Jonkoping                                   
                             University\JTH\Programmering\Python\enhetscirkeln till                                
                             sincosfunk\media\videos\enhetscirkeln till                                            
                             sincosfunk\720p30\SinusUtrullning.mp4'                                                
                                                                                                                   

                    INFO     The segment cache exceeded 100 files; removed 31 least recently used     ]8;id=15997581;file://c:\Users\sanpar\AppData\Local\Programs\Python\Python312\Lib\site-packages\manim\utils\caching.py\caching.py]8;;\:]8;id=15997582;file://c:\Users\sanpar\AppData\Local\Programs\Python\Python312\Lib\site-packages\manim\utils\caching.py#61\61]8;;\
                             segment(s). Change max_files_cached to adjust this limit.                             

                    INFO     Rendered SinusUtrullning                                                ]8;id=15997587;file://c:\Users\sanpar\AppData\Local\Programs\Python\Python312\Lib\site-packages\manim\manager.py\manager.py]8;;\:]8;id=15997588;file://c:\Users\sanpar\AppData\Local\Programs\Python\Python312\Lib\site-packages\manim\manager.py#226\226]8;;\
                             Played 39 animations                                                                  

[11/04/26 16:00:23] INFO     Previewed File at: 'c:\Users\sanpar\OneDrive - Jonkoping               ]8;id=15997593;file://c:\Users\sanpar\AppData\Local\Programs\Python\Python312\Lib\site-packages\manim\utils\file_ops.py\file_ops.py]8;;\:]8;id=15997594;file://c:\Users\sanpar\AppData\Local\Programs\Python\Python312\Lib\site-packages\manim\utils\file_ops.py#143\143]8;;\
                             University\JTH\Programmering\Python\enhetscirkeln till                                
                             sincosfunk\media\videos\enhetscirkeln till                                            
                             sincosfunk\720p30\SinusUtrullning.mp4'                                                